# Targeted Dropout Uncertainty Ladder on Colab

This notebook runs the Stage A targeted-dropout ladder for `TASK_targeted_dropout_uncertainty_ladder.md`. It creates a balanced 24,960-row proof set from the official 500K score pool, scores prior and Books models for each targeted dropout config, persists raw MC samples, and writes strategy metrics to Drive.

Replace `OLMO_SHA` with the pushed commit containing this notebook, `scripts/21_dropout_uncertainty_metrics.py`, `scripts/22_dropout_strategy_sweep.py`, and `configs/sweeps/score-targeted-dropout-uncertainty.yaml` before GPU work.

## 0. Resource Assumptions

- GPU: A100 preferred; L4/T4 may require smaller microbatch.
- System RAM: high-RAM Colab recommended for token subset and Parquet writes.
- Drive quota: at least 10 GB free for Stage A raw shards, MC samples, metrics, and logs.
- Remote downloads: none if the official score-pool artifacts and native OLMo checkpoints already exist on Drive.
- Stage A size: 24,960 rows, balanced as 4,992 rows from each of the five score-pool groups, so it is close to 25K and divisible by batch size 32.

## 1. Runtime and Drive

One-time setup. Check GPU and mount Drive before referencing Drive paths.

In [ ]:
# PYTHON CELL
!nvidia-smi


In [ ]:
# PYTHON CELL
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# PYTHON CELL
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/color-filter-ablation')
DATA_DRIVE = DRIVE / 'data'
MODELS_DRIVE = DRIVE / 'assets' / 'raw' / 'models'
SOURCE_RESULTS_DRIVE = DRIVE / 'results'
RESULTS_DRIVE = DRIVE / 'results' / 'dropout-uncertainty-targeted'
SUBSET_ID = 'stage_a_25k'
STAGE_ROOT = RESULTS_DRIVE / SUBSET_ID
RAW_SCORE_DRIVE = STAGE_ROOT / 'raw_score_shards'
CONFIG_DRIVE = DRIVE / 'runtime_configs' / 'dropout-uncertainty-targeted' / SUBSET_ID

for path in [RESULTS_DRIVE, STAGE_ROOT, RAW_SCORE_DRIVE, CONFIG_DRIVE]:
    path.mkdir(parents=True, exist_ok=True)
if not DRIVE.exists():
    raise FileNotFoundError(DRIVE)
print('drive root:', DRIVE)
print('stage root:', STAGE_ROOT)
print('raw score root:', RAW_SCORE_DRIVE)
print('runtime configs:', CONFIG_DRIVE)


In [ ]:
# PYTHON CELL
!df -h /content /content/drive/MyDrive


## 2. Clone, Pin, and Install

One-time setup. This cell stops if `OLMO_SHA` is still the placeholder.

In [ ]:
# PYTHON CELL
import subprocess
from pathlib import Path

OLMO_DIR = Path('/content/color-filter-olmo')
OLMO_REPO = 'https://github.com/myazdani/color-filter-olmo.git'
OLMO_SHA = 'REPLACE_WITH_PUSHED_COMMIT_SHA'

def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

def out(*args, cwd=None):
    return subprocess.check_output([str(arg) for arg in args], cwd=cwd, text=True).strip()

if OLMO_SHA == 'REPLACE_WITH_PUSHED_COMMIT_SHA':
    raise RuntimeError('Set OLMO_SHA to the pushed commit before running Colab GPU work.')
if not OLMO_DIR.exists():
    run('git', 'clone', OLMO_REPO, OLMO_DIR)
elif (OLMO_DIR / '.git').is_dir():
    run('git', '-C', OLMO_DIR, 'fetch', 'origin')
else:
    raise RuntimeError(f'{OLMO_DIR} exists but is not a git checkout')
run('git', '-C', OLMO_DIR, 'checkout', OLMO_SHA)
actual = out('git', '-C', OLMO_DIR, 'rev-parse', 'HEAD')
if actual != OLMO_SHA:
    raise RuntimeError(f'SHA mismatch: expected {OLMO_SHA}, got {actual}')
print('OLMo runtime SHA:', actual)


In [ ]:
# PYTHON CELL
import importlib
import subprocess
import torch

overlay = [
    'omegaconf==2.3.0', 'rich', 'tokenizers', 'transformers',
    'cached_path==1.8.10', 'packaging', 'boto3', 'google-cloud-storage',
    'wandb', 'torchmetrics', 'datasets', 'huggingface_hub', 'matplotlib',
    'markdown', 'pyyaml', 'pandas', 'pyarrow'
]
subprocess.run(['python', '-m', 'pip', 'install', '-q', *overlay], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', str(OLMO_DIR), '--no-deps'], check=True)
for module_name in ['numpy', 'pandas', 'pyarrow', 'yaml', 'omegaconf', 'torchmetrics', 'transformers']:
    importlib.import_module(module_name)
for path in [
    OLMO_DIR / 'scripts/train.py',
    OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py',
    OLMO_DIR / 'scripts/22_dropout_strategy_sweep.py',
    OLMO_DIR / 'configs/sweeps/score-targeted-dropout-uncertainty.yaml',
]:
    if not path.exists():
        raise FileNotFoundError(path)
    print('ok:', path)
subprocess.run(['python', str(OLMO_DIR / 'scripts/21_dropout_uncertainty_metrics.py'), '--help'], check=True, stdout=subprocess.PIPE)
subprocess.run(['python', str(OLMO_DIR / 'scripts/22_dropout_strategy_sweep.py'), '--help'], check=True, stdout=subprocess.PIPE)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda, 'gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


## 3. Configure Paths and Target Ladder

Safe to rerun. The target list mirrors the task file and template YAML.

In [ ]:
# PYTHON CELL
from pathlib import Path

TOKENS_DRIVE = DATA_DRIVE / 'score_pool_tokens_official_500k.npy'
META_DRIVE = DATA_DRIVE / 'score_pool_meta_official_500k.parquet'
FULL_SCORES_DRIVE = SOURCE_RESULTS_DRIVE / 'score-pool-robustness-official-500k' / 'scores_full.parquet'
PRIOR_CHECKPOINT = MODELS_DRIVE / 'prior'
BOOKS_CHECKPOINT = MODELS_DRIVE / 'conditional_books'

NUM_SAMPLES = 8
SEED = 1
SEQ_LEN = 512
GLOBAL_BATCH_SIZE = 32
MICROBATCH = 32
ROWS_PER_POOL = 4_992
STAGE_ROWS = ROWS_PER_POOL * 5
SMOKE_ROWS_PER_POOL = 64
SMOKE_ROWS = SMOKE_ROWS_PER_POOL * 5
TAU64_CUTOFF = 0.3513622284

TARGET_CONFIGS = [
    {'config_id': 'dropout_trainmode_p000', 'dropout_target': 'none', 'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.0},
    {'config_id': 'dropout_attn_p001', 'dropout_target': 'attention', 'attention_dropout': 0.01, 'residual_dropout': 0.0, 'embedding_dropout': 0.0},
    {'config_id': 'dropout_attn_p002', 'dropout_target': 'attention', 'attention_dropout': 0.02, 'residual_dropout': 0.0, 'embedding_dropout': 0.0},
    {'config_id': 'dropout_resid_p001', 'dropout_target': 'residual', 'attention_dropout': 0.0, 'residual_dropout': 0.01, 'embedding_dropout': 0.0},
    {'config_id': 'dropout_resid_p002', 'dropout_target': 'residual', 'attention_dropout': 0.0, 'residual_dropout': 0.02, 'embedding_dropout': 0.0},
    {'config_id': 'dropout_embed_p0005', 'dropout_target': 'embedding', 'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.005},
    {'config_id': 'dropout_embed_p001', 'dropout_target': 'embedding', 'attention_dropout': 0.0, 'residual_dropout': 0.0, 'embedding_dropout': 0.01},
    {'config_id': 'dropout_attn_resid_p001', 'dropout_target': 'attention+residual', 'attention_dropout': 0.01, 'residual_dropout': 0.01, 'embedding_dropout': 0.0},
]
print('stage rows:', STAGE_ROWS)
print('target configs:', [c['config_id'] for c in TARGET_CONFIGS])


## 4. Validate Inputs and Build Balanced Subset

Safe to rerun. This creates subset token, metadata, and full-score files under `/content`, then writes a manifest to Drive.

In [ ]:
# PYTHON CELL
import json
from pathlib import Path
import numpy as np
import pandas as pd

for path in [TOKENS_DRIVE, META_DRIVE, FULL_SCORES_DRIVE, PRIOR_CHECKPOINT / 'model.pt', PRIOR_CHECKPOINT / 'config.yaml', BOOKS_CHECKPOINT / 'model.pt', BOOKS_CHECKPOINT / 'config.yaml']:
    if not path.exists():
        raise FileNotFoundError(path)
    print('ok:', path)

tokens = np.load(TOKENS_DRIVE, mmap_mode='r')
meta = pd.read_parquet(META_DRIVE)
full_scores = pd.read_parquet(FULL_SCORES_DRIVE)
if tokens.shape[0] != len(meta) or len(meta) != len(full_scores):
    raise ValueError((tokens.shape, len(meta), len(full_scores)))
if tokens.shape[1] != SEQ_LEN:
    raise ValueError(tokens.shape)
if 'pool_name' not in meta.columns:
    raise ValueError('metadata must contain pool_name')

rng = np.random.default_rng(SEED)
selected = []
pool_counts = meta['pool_name'].value_counts().sort_index().to_dict()
print('source pool counts:', pool_counts)
for pool in sorted(pool_counts):
    idx = np.flatnonzero(meta['pool_name'].to_numpy() == pool)
    if len(idx) < ROWS_PER_POOL:
        raise ValueError(f'Pool {pool} only has {len(idx)} rows')
    chosen = np.sort(rng.choice(idx, size=ROWS_PER_POOL, replace=False))
    selected.append(chosen)
selected_idx = np.concatenate(selected).astype(np.int64)
if len(selected_idx) != STAGE_ROWS or STAGE_ROWS % GLOBAL_BATCH_SIZE != 0:
    raise ValueError((len(selected_idx), STAGE_ROWS, GLOBAL_BATCH_SIZE))

SUBSET_LOCAL = Path('/content/targeted_dropout_stage_a_subset')
SUBSET_LOCAL.mkdir(parents=True, exist_ok=True)
subset_token_npy = SUBSET_LOCAL / f'{SUBSET_ID}_tokens.npy'
subset_meta = SUBSET_LOCAL / f'{SUBSET_ID}_meta.parquet'
subset_full = SUBSET_LOCAL / f'{SUBSET_ID}_full_scores.parquet'
subset_raw = SUBSET_LOCAL / f'{SUBSET_ID}_tokens.uint32.raw'

if not subset_token_npy.exists():
    arr = np.asarray(tokens[selected_idx], dtype=np.uint32)
    np.save(subset_token_npy, arr)
    arr.tofile(subset_raw)
else:
    arr = np.load(subset_token_npy, mmap_mode='r')
    if not subset_raw.exists():
        np.asarray(arr, dtype=np.uint32).tofile(subset_raw)

meta_subset = meta.iloc[selected_idx].reset_index(drop=True).copy()
meta_subset['source_row'] = selected_idx
full_subset = full_scores.iloc[selected_idx].reset_index(drop=True).copy()
meta_subset.to_parquet(subset_meta, index=False)
full_subset.to_parquet(subset_full, index=False)

SUBSET_MANIFEST = STAGE_ROOT / f'{SUBSET_ID}_manifest.json'
SUBSET_MANIFEST.write_text(json.dumps({
    'subset_id': SUBSET_ID,
    'row_count': int(STAGE_ROWS),
    'rows_per_pool': int(ROWS_PER_POOL),
    'seed': int(SEED),
    'pool_counts': meta_subset['pool_name'].value_counts().sort_index().to_dict(),
    'source_token_path': str(TOKENS_DRIVE),
    'source_metadata_path': str(META_DRIVE),
    'source_full_scores_path': str(FULL_SCORES_DRIVE),
    'subset_token_npy': str(subset_token_npy),
    'subset_raw': str(subset_raw),
    'subset_metadata': str(subset_meta),
    'subset_full_scores': str(subset_full),
}, indent=2, sort_keys=True) + '\n')
print('subset tokens:', arr.shape, arr.dtype)
print('subset pool counts:', meta_subset['pool_name'].value_counts().sort_index().to_dict())
print('manifest:', SUBSET_MANIFEST)


## 5. Scoring Helpers

Safe to rerun. Builds runtime configs from the target template and skips already completed valid score directories.

In [ ]:
# PYTHON CELL
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
import torch
from omegaconf import OmegaConf

sys.path.insert(0, str(OLMO_DIR))
from olmo.config import TrainConfig

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')
TEMPLATE_CONFIG = OLMO_DIR / 'configs/sweeps/score-targeted-dropout-uncertainty.yaml'
RUNTIME_CHECKPOINT_DIR = Path('/content/targeted_dropout_runtime_checkpoints')
RUNTIME_CONFIG_DIR = Path('/content/targeted_dropout_runtime_configs')
RUNTIME_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def symlink_or_refresh(src, dst):
    src = Path(src); dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        try:
            if dst.resolve() == src.resolve():
                return
        except FileNotFoundError:
            pass
        dst.unlink()
    os.symlink(src, dst)

def prepare_model_only_checkpoint(checkpoint_path):
    source = Path(checkpoint_path)
    runtime = RUNTIME_CHECKPOINT_DIR / source.name
    runtime.mkdir(parents=True, exist_ok=True)
    symlink_or_refresh(source / 'model.pt', runtime / 'model.pt')
    symlink_or_refresh(source / 'config.yaml', runtime / 'config.yaml')
    for state_name in ['train.pt', 'other.pt']:
        path = runtime / state_name
        if not path.exists():
            torch.save({}, path)
    return runtime

def load_checkpoint_score_config(checkpoint_path):
    checkpoint_cfg = OmegaConf.load(Path(checkpoint_path) / 'config.yaml')
    cfg = OmegaConf.load(TEMPLATE_CONFIG)
    cfg.model = checkpoint_cfg.model
    if 'tokenizer' in checkpoint_cfg:
        cfg.tokenizer = checkpoint_cfg.tokenizer
    if 'targeted_ladder' in cfg:
        del cfg['targeted_ladder']
    return cfg

def build_score_config(config, model_id, checkpoint_path, output_dir, rows, data_start_step=0, microbatch=MICROBATCH):
    if rows % GLOBAL_BATCH_SIZE != 0:
        raise ValueError(f'rows={rows} must be divisible by {GLOBAL_BATCH_SIZE}')
    cfg = load_checkpoint_score_config(checkpoint_path)
    cfg.run_name = f"{config['config_id']}_{model_id}"
    cfg.save_folder = str(output_dir)
    cfg.load_path = str(prepare_model_only_checkpoint(checkpoint_path))
    cfg.max_duration = rows // GLOBAL_BATCH_SIZE
    cfg.data_start_step = data_start_step
    cfg.global_train_batch_size = GLOBAL_BATCH_SIZE
    cfg.device_train_batch_size = GLOBAL_BATCH_SIZE
    cfg.device_train_microbatch_size = microbatch
    cfg.data.paths = [str(subset_raw)]
    cfg.data.memmap_dtype = 'uint32'
    cfg.seed = SEED
    cfg.model.attention_dropout = float(config['attention_dropout'])
    cfg.model.residual_dropout = float(config['residual_dropout'])
    cfg.model.embedding_dropout = float(config['embedding_dropout'])
    cfg.uncertainty_scoring.enabled = True
    cfg.uncertainty_scoring.num_samples = NUM_SAMPLES
    cfg.uncertainty_scoring.perturbation_type = 'dropout'
    cfg.uncertainty_scoring.coupled_masks = True
    cfg.restore_dataloader = False
    cfg.reset_optimizer_state = True
    cfg.reset_trainer_state = True
    return cfg

def write_config(cfg, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    OmegaConf.save(cfg, path)
    TrainConfig.load(str(path), validate_paths=False)
    drive_path = CONFIG_DRIVE / path.name
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, drive_path)
    print('wrote runtime config:', path, 'drive copy:', drive_path)
    return path

def run_logged(args, log_path, cwd=OLMO_DIR, append=False):
    env = os.environ.copy()
    env['PYTHONPATH'] = str(OLMO_DIR) + (os.pathsep + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    mode = 'a' if append else 'w'
    print('running:', ' '.join(str(a) for a in args))
    start = time.perf_counter()
    with log_path.open(mode, encoding='utf-8') as log:
        proc = subprocess.Popen([str(a) for a in args], cwd=str(cwd), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
        ret = proc.wait()
    elapsed = time.perf_counter() - start
    print('elapsed_seconds:', round(elapsed, 2), 'log:', log_path)
    if ret != 0:
        raise RuntimeError(f'Command failed with code {ret}. Tail:\n{log_path.read_text(errors="ignore")[-4000:]}')

def score_index_complete(score_dir, expected_rows):
    index_path = score_dir / 'mmap_index.npy'
    if not index_path.exists():
        return False
    # Drive-backed mmap_index.npy is preallocated to 1e8 entries, so scanning
    # backwards is slow. This Stage A run should write one unique row index for
    # every subset row; reading only the expected prefix is enough to catch
    # interrupted runs where the unwritten tail remains zero-filled.
    idx = np.memmap(index_path, dtype=np.int64, mode='r')
    if len(idx) < expected_rows:
        return False
    head = np.asarray(idx[:expected_rows], dtype=np.int64)
    return len(np.unique(head)) == expected_rows


def infer_score_rows(score_dir):
    return STAGE_ROWS if score_index_complete(score_dir, STAGE_ROWS) else 0

def score_width(score_dir):
    files_txt = score_dir / 'files.txt'
    if not files_txt.exists():
        return 0
    files = [line.strip() for line in files_txt.read_text().splitlines() if line.strip()]
    if not files:
        return 0
    path = Path(files[0])
    if not path.exists():
        path = score_dir / path.name
    return (path.stat().st_size // np.dtype(np.float32).itemsize) // 1_048_576

def valid_score_dir(score_dir, expected_rows):
    score_dir = Path(score_dir)
    return (
        (score_dir / 'files.txt').exists()
        and (score_dir / 'mmap_index.npy').exists()
        and score_width(score_dir) >= NUM_SAMPLES
        and score_index_complete(score_dir, expected_rows)
    )

def run_score_once(config, model_id, checkpoint_path, output_dir, rows, microbatch=MICROBATCH):
    score_dir = output_dir / 'score'
    if valid_score_dir(score_dir, rows):
        print('skip valid score dir:', score_dir)
        return score_dir
    if output_dir.exists() and not valid_score_dir(score_dir, rows):
        print('removing invalid isolated output:', output_dir)
        shutil.rmtree(output_dir)
    cfg_path = RUNTIME_CONFIG_DIR / f"{config['config_id']}_{model_id}.yaml"
    cfg = build_score_config(config, model_id, checkpoint_path, output_dir, rows, microbatch=microbatch)
    write_config(cfg, cfg_path)
    log_path = output_dir.with_suffix('.log')
    run_logged(['torchrun', '--standalone', '--nproc_per_node=1', 'scripts/train.py', cfg_path, '--save_overwrite=true'], log_path)
    if not valid_score_dir(score_dir, rows):
        raise RuntimeError(f'Invalid score output: {score_dir}')
    return score_dir

probe = build_score_config(TARGET_CONFIGS[0], 'prior_probe', PRIOR_CHECKPOINT, STAGE_ROOT / 'probe', GLOBAL_BATCH_SIZE)
probe_path = write_config(probe, RUNTIME_CONFIG_DIR / 'probe.yaml')
print('probe config ok:', probe_path)


## 6. Smoke Test / GPU Gate

Benchmark only. Scores a 320-row balanced subset for the train-mode zero-dropout control. Stop if this fails.

In [ ]:
# PYTHON CELL
SMOKE_CONFIG = TARGET_CONFIGS[0]
SMOKE_ROOT = STAGE_ROOT / 'smoke' / SMOKE_CONFIG['config_id']
smoke_prior = run_score_once(SMOKE_CONFIG, 'prior', PRIOR_CHECKPOINT, SMOKE_ROOT / 'prior', SMOKE_ROWS)
smoke_books = run_score_once(SMOKE_CONFIG, 'books', BOOKS_CHECKPOINT, SMOKE_ROOT / 'books', SMOKE_ROWS)
SMOKE_ANALYSIS = SMOKE_ROOT / 'analysis'
run_logged([
    'python', 'scripts/21_dropout_uncertainty_metrics.py',
    '--prior-score-dir', smoke_prior,
    '--conditional-score-dir', smoke_books,
    '--output-dir', SMOKE_ANALYSIS,
    '--config-id', f"{SMOKE_CONFIG['config_id']}_smoke",
    '--metadata', subset_meta,
    '--full-scores', subset_full,
    '--num-samples', NUM_SAMPLES,
    '--max-rows', SMOKE_ROWS,
    '--dropout-target', SMOKE_CONFIG['dropout_target'],
    '--attention-dropout', SMOKE_CONFIG['attention_dropout'],
    '--residual-dropout', SMOKE_CONFIG['residual_dropout'],
    '--embedding-dropout', SMOKE_CONFIG['embedding_dropout'],
    '--seed', SEED,
], SMOKE_ANALYSIS / 'aggregate.log')
raw = np.load(SMOKE_ANALYSIS / f"mc_samples_{SMOKE_CONFIG['config_id']}_smoke.npz")
color_samples = raw['color_samples']
assert color_samples.shape == (SMOKE_ROWS, NUM_SAMPLES), color_samples.shape
assert np.isfinite(color_samples).all()
print('smoke color shape:', color_samples.shape, 'mean std:', float(color_samples.std(axis=1, ddof=1).mean()))


## 7. Batch-Size and Shard-Size Tuning

This bounded benchmark runs only on the smoke subset and writes to an isolated `benchmark/` directory. It selects the largest successful microbatch for the full Stage A run. Stage A itself remains a single resumable shard per model/config (`STAGE_ROWS = 24,960`) so each config can be skipped independently after Colab disconnects.


In [ ]:
# PYTHON CELL
BENCH_CONFIG = TARGET_CONFIGS[0]
BENCH_ROOT = STAGE_ROOT / 'benchmark' / BENCH_CONFIG['config_id']
MICROBATCH_CANDIDATES = [16, 32]
benchmark_results = []
selected_microbatch = None

for candidate in MICROBATCH_CANDIDATES:
    out_dir = BENCH_ROOT / f'prior_microbatch_{candidate}'
    try:
        run_score_once(BENCH_CONFIG, 'prior', PRIOR_CHECKPOINT, out_dir, SMOKE_ROWS, microbatch=candidate)
        benchmark_results.append({'microbatch': candidate, 'status': 'ok', 'output_dir': str(out_dir)})
        selected_microbatch = candidate
    except Exception as exc:
        benchmark_results.append({'microbatch': candidate, 'status': f'failed: {exc}', 'output_dir': str(out_dir)})
        print('stopping benchmark ladder after first failure')
        break

if selected_microbatch is None:
    raise RuntimeError(f'No successful microbatch candidate: {benchmark_results}')

MICROBATCH = selected_microbatch
SHARD_ROWS = STAGE_ROWS
print('benchmark results:', benchmark_results)
print('selected MICROBATCH:', MICROBATCH)
print('Stage A shard rows:', SHARD_ROWS, '(single shard per model/config)')


## 8. Full Resumable Stage A Run


In [ ]:
# PYTHON CELL
stage_results = []
for config in TARGET_CONFIGS:
    config_id = config['config_id']
    root = RAW_SCORE_DRIVE / config_id
    prior_dir = run_score_once(config, 'prior', PRIOR_CHECKPOINT, root / 'prior', STAGE_ROWS)
    books_dir = run_score_once(config, 'books', BOOKS_CHECKPOINT, root / 'books', STAGE_ROWS)
    stage_results.append({'config': config, 'prior_dir': prior_dir, 'books_dir': books_dir})
print('completed configs:', [r['config']['config_id'] for r in stage_results])


## 9. Resume After Disconnect / Status

Safe to rerun. After reconnect, rerun Sections 1-5, then this status cell. Rerunning Section 8 skips valid score outputs.


In [ ]:
# PYTHON CELL
status = []
stage_results = []
for config in TARGET_CONFIGS:
    config_status = {'config': config, 'valid_models': {}}
    for model_id in ['prior', 'books']:
        score_dir = RAW_SCORE_DRIVE / config['config_id'] / model_id / 'score'
        valid = valid_score_dir(score_dir, STAGE_ROWS) if score_dir.exists() else False
        config_status['valid_models'][model_id] = valid
        status.append({
            'config_id': config['config_id'],
            'model': model_id,
            'exists': score_dir.exists(),
            'valid': valid,
            'score_dir': str(score_dir),
        })
    if config_status['valid_models'].get('prior') and config_status['valid_models'].get('books'):
        root = RAW_SCORE_DRIVE / config['config_id']
        stage_results.append({
            'config': config,
            'prior_dir': root / 'prior' / 'score',
            'books_dir': root / 'books' / 'score',
        })

complete = sum(row['valid'] for row in status)
print('valid score dirs:', complete, '/', len(status))
print('reconstructed complete configs:', len(stage_results), '/', len(TARGET_CONFIGS))
if len(stage_results) != len(TARGET_CONFIGS):
    print('Rerun Section 8 to score missing configs; it skips valid score dirs.')
for row in status:
    print(row)


## 10. Metrics, Report, and Outputs to Bring Back

Safe to rerun after scoring. Writes compact raw MC artifacts and strategy metrics directly under Drive.


In [ ]:
# PYTHON CELL
EXPECTED_SELECTED_FILES = 64


def analysis_is_complete(config_id):
    analysis = STAGE_ROOT / config_id / 'analysis'
    strategy = analysis / 'strategy'
    required = [
        analysis / 'aggregate.log',
        analysis / f'mc_samples_{config_id}.npz',
        analysis / f'mc_samples_{config_id}.parquet',
        analysis / f'mc_samples_{config_id}_manifest.json',
        analysis / 'color_distribution_summary.parquet',
        analysis / 'strategy.log',
        strategy / 'strategy_sweep_metrics.csv',
        strategy / 'strategy_selection_overlap.csv',
    ]
    selected_files = list((strategy / 'strategy_selected_indices').glob('*.npy'))
    return all(path.exists() for path in required) and len(selected_files) == EXPECTED_SELECTED_FILES


analysis_dirs = []
complete_config_ids = {item['config']['config_id'] for item in stage_results}
missing_score_configs = [config['config_id'] for config in TARGET_CONFIGS if config['config_id'] not in complete_config_ids]
if missing_score_configs:
    raise RuntimeError(
        'Raw scoring is incomplete for these configs: '
        + ', '.join(missing_score_configs)
        + '. Rerun Section 9 for status, then Section 8 to resume missing score dirs, then rerun Section 10.'
    )

for item in stage_results:
    config = item['config']
    config_id = config['config_id']
    analysis = STAGE_ROOT / config_id / 'analysis'
    analysis.mkdir(parents=True, exist_ok=True)
    if analysis_is_complete(config_id):
        print('skip complete analysis:', config_id)
        analysis_dirs.append(analysis)
        continue
    run_logged([
        'python', 'scripts/21_dropout_uncertainty_metrics.py',
        '--prior-score-dir', item['prior_dir'],
        '--conditional-score-dir', item['books_dir'],
        '--output-dir', analysis,
        '--config-id', config_id,
        '--metadata', subset_meta,
        '--full-scores', subset_full,
        '--num-samples', NUM_SAMPLES,
        '--max-rows', STAGE_ROWS,
        '--dropout-target', config['dropout_target'],
        '--attention-dropout', config['attention_dropout'],
        '--residual-dropout', config['residual_dropout'],
        '--embedding-dropout', config['embedding_dropout'],
        '--seed', SEED,
    ], analysis / 'aggregate.log')
    run_logged([
        'python', 'scripts/22_dropout_strategy_sweep.py',
        '--mc-samples', analysis / f'mc_samples_{config_id}.npz',
        '--summary', analysis / 'color_distribution_summary.parquet',
        '--output-dir', analysis / 'strategy',
        '--tau64-cutoff', TAU64_CUTOFF,
    ], analysis / 'strategy.log')
    selected_count = len(list((analysis / 'strategy' / 'strategy_selected_indices').glob('*.npy')))
    if selected_count != EXPECTED_SELECTED_FILES:
        raise RuntimeError(f"{config_id}: strategy sweep wrote {selected_count} selected-index files")
    analysis_dirs.append(analysis)
print('analysis dirs:')
for path in analysis_dirs:
    print(path)


## 11. Output Review, Acceptance Checks, and Local Download Bundle

Safe to rerun after aggregation. Completion requires these files on Drive. The following cells verify report readiness and create a compact local-download bundle.


In [ ]:
# PYTHON CELL
import numpy as np
import pandas as pd

expected = []
selection_counts = []
review_rows = []
for config in TARGET_CONFIGS:
    analysis = STAGE_ROOT / config['config_id'] / 'analysis'
    strategy = analysis / 'strategy'
    expected.extend([
        analysis / 'aggregate.log',
        analysis / f"mc_samples_{config['config_id']}.npz",
        analysis / f"mc_samples_{config['config_id']}.parquet",
        analysis / f"mc_samples_{config['config_id']}_manifest.json",
        analysis / 'color_distribution_summary.parquet',
        analysis / 'strategy.log',
        strategy / 'strategy_sweep_metrics.csv',
        strategy / 'strategy_selection_overlap.csv',
    ])
    selected_files = sorted((strategy / 'strategy_selected_indices').glob('*.npy'))
    selection_counts.append((config['config_id'], len(selected_files)))

missing = [str(p) for p in expected if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing expected analysis outputs. Rerun Section 9 for raw-score status, Section 8 if any score dirs are missing, then Section 10.\n'
        + '\n'.join(missing)
    )

for config_id, count in selection_counts:
    if count != 64:
        raise RuntimeError(f"{config_id} expected 64 selected-index files, found {count}. Rerun Section 10.")

for config in TARGET_CONFIGS:
    cid = config['config_id']
    analysis = STAGE_ROOT / cid / 'analysis'
    raw = np.load(analysis / f"mc_samples_{cid}.npz")
    color = raw['color_samples']
    utility = raw['utility_samples']
    if color.shape != (STAGE_ROWS, NUM_SAMPLES):
        raise RuntimeError(f"{cid}: unexpected color sample shape {color.shape}")
    if utility.shape != (STAGE_ROWS, NUM_SAMPLES):
        raise RuntimeError(f"{cid}: unexpected utility sample shape {utility.shape}")
    if not np.isfinite(color).all() or not np.isfinite(utility).all():
        raise RuntimeError(f"{cid}: non-finite values in MC samples")

    summary = pd.read_parquet(analysis / 'color_distribution_summary.parquet')
    metrics = pd.read_csv(analysis / 'strategy' / 'strategy_sweep_metrics.csv')
    scopes = metrics['metric_scope'].value_counts().to_dict()
    if len(summary) != STAGE_ROWS:
        raise RuntimeError(f"{cid}: summary rows {len(summary)} != {STAGE_ROWS}")
    if scopes.get('pairwise') != 96 or scopes.get('full_pool') != 64:
        raise RuntimeError(f"{cid}: unexpected metric scopes {scopes}")

    mc_std_mean = float(color.std(axis=1, ddof=1).mean())
    if cid == 'dropout_trainmode_p000' and mc_std_mean > 1e-5:
        raise RuntimeError(f"{cid}: zero-dropout control is not deterministic enough; mean MC std={mc_std_mean}")
    review_rows.append({
        'config_id': cid,
        'rows': len(summary),
        'sample_shape': tuple(color.shape),
        'mean_row_mc_std': mc_std_mean,
        'metric_rows': len(metrics),
        'metric_scopes': scopes,
    })

pd.DataFrame(review_rows)


### Local Download Bundle

This cell creates a compact zip for local report writing. It includes compact MC samples, summaries, strategy metrics, selection indices, logs, manifests, and runtime configs. It intentionally excludes checkpoints, raw model weights, token arrays, and raw score mmap files.


In [ ]:
# PYTHON CELL
import json
import zipfile
from datetime import datetime, timezone
from google.colab import files

AUTO_DOWNLOAD = True
archive_path = Path('/content') / f'targeted_dropout_ladder_{SUBSET_ID}.zip'
manifest_path = Path('/content') / f'targeted_dropout_ladder_{SUBSET_ID}_bundle_manifest.json'

bundle_manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'subset_id': SUBSET_ID,
    'stage_rows': STAGE_ROWS,
    'num_samples': NUM_SAMPLES,
    'selected_microbatch': MICROBATCH,
    'olmo_sha': OLMO_SHA,
    'stage_root': str(STAGE_ROOT),
    'analysis_root_pattern': str(STAGE_ROOT / '<config_id>' / 'analysis'),
    'runtime_config_drive': str(CONFIG_DRIVE),
    'target_configs': TARGET_CONFIGS,
    'local_report_command': (
        'conda run -n ml python color-filter-ablation/scripts/20_targeted_dropout_ladder_report.py '
        '--stage-root /path/to/unzipped/results/dropout-uncertainty-targeted/stage_a_25k '
        '--output-dir color-filter-ablation/reports/dropout-uncertainty/targeted-ladder'
    ),
    'exclusions': ['checkpoints', 'raw model weights', 'token arrays', 'raw score mmap files'],
}
manifest_path.write_text(json.dumps(bundle_manifest, indent=2, sort_keys=True))

def add_if_exists(zf, path, arcname=None):
    path = Path(path)
    if not path.exists() or path.is_dir():
        return False
    if arcname is None:
        try:
            arcname = str(path.relative_to(DRIVE))
        except ValueError:
            arcname = path.name
    zf.write(path, arcname)
    return True

with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    zf.write(manifest_path, 'bundle_manifest.json')
    add_if_exists(zf, SUBSET_MANIFEST, 'subset_manifest.json')

    for cfg_path in sorted(CONFIG_DRIVE.glob('*.yaml')):
        add_if_exists(zf, cfg_path, f'runtime_configs/{cfg_path.name}')

    for config in TARGET_CONFIGS:
        cid = config['config_id']
        analysis = STAGE_ROOT / cid / 'analysis'
        strategy = analysis / 'strategy'
        base = f'results/dropout-uncertainty-targeted/{SUBSET_ID}/analysis/{cid}'

        for name in [
            'aggregate.log',
            'strategy.log',
            'color_distribution_summary.parquet',
            f'mc_samples_{cid}.npz',
            f'mc_samples_{cid}.parquet',
            f'mc_samples_{cid}_manifest.json',
        ]:
            add_if_exists(zf, analysis / name, f'{base}/{name}')

        for name in ['strategy_sweep_metrics.csv', 'strategy_selection_overlap.csv']:
            add_if_exists(zf, strategy / name, f'{base}/strategy/{name}')

        selected_dir = strategy / 'strategy_selected_indices'
        for selected in sorted(selected_dir.glob('*.npy')):
            add_if_exists(zf, selected, f'{base}/strategy/strategy_selected_indices/{selected.name}')

        raw_root = RAW_SCORE_DRIVE / cid
        for model_id in ['prior', 'books']:
            raw_base = f'results/dropout-uncertainty-targeted/{SUBSET_ID}/raw_score_shards/{cid}/{model_id}'
            add_if_exists(zf, raw_root / f'{model_id}.log', f'{raw_base}.log')
            add_if_exists(zf, raw_root / model_id / 'config.yaml', f'{raw_base}/config.yaml')
            add_if_exists(zf, raw_root / model_id / 'score' / 'files.txt', f'{raw_base}/score/files.txt')

size_mb = archive_path.stat().st_size / 1e6
print('wrote bundle:', archive_path, 'size_mb=', round(size_mb, 2))
if AUTO_DOWNLOAD:
    files.download(str(archive_path))
